# Study v3 — single test-partition evaluation

Run once, after the freeze tag `study-v3-frozen`, on a T4 GPU runtime. It loads the 30
selected checkpoints from Drive (no training), predicts the official test partitions
with the RAW and STD defenses, and writes a text-free ZIP to `NASN/study-v3/test-analysis/`.
Finished models are skipped, so a disconnected session resumes with Run all
(about 2.5 hours in total).

In [ ]:
REF = '6132d3de9efac561e55cc5b7685113e31fafcf71'
DRIVE = '/content/drive/MyDrive/NASN/study-v3'
FRESH_START = False
DATASETS = ['arsarcasm', 'arsas']
ENCODERS = ['MARBERT', 'MARBERTv2', 'AraBERTv02-twitter', 'CAMeLBERT-DA', 'XLM-R-base']
SEEDS = [42, 43, 44]
TEST_DEFENSES = ['RAW', 'STD']
TEST_VARIANTS_CONTENT_SHA256 = {"arsarcasm": "58c10133f57f06c390aaf3ef299cac93eeffaa70672a0ee0567b4ac959965bc0", "arsas": "6eca44775bb31ac717f9cd301e28abfe0492997a508d216b62736bf395c992a6"}

In [ ]:
import gzip, hashlib, json, os, subprocess, sys, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/nasn-study-v3')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/haneenbiate/-.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', REF], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-marbert.txt', 'accelerate==1.10.1'], cwd=REPO, check=True)
# Colab preinstalls torchvision/torchaudio built for its own torch; with the pinned torch they fail to
# load ("torchvision::nms does not exist") and transformers then cannot import BERT. Text only: remove them.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchvision', 'torchaudio'], check=True)
ROOT = Path(DRIVE)
# Guard against silently starting over, e.g. from another Google account without the shared folder.
if not FRESH_START and not (ROOT / 'arsarcasm' / 'train').exists():
    raise AssertionError(f'No earlier results in {DRIVE}. From another Google account: share the NASN folder '
                         'with it and add a shortcut to My Drive (Shared with me > NASN > Organize > Add shortcut). '
                         'To start from scratch on purpose, set FRESH_START = True in the first cell.')
ROOT.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, 'PYTHONUTF8': '1', 'HF_HUB_DISABLE_XET': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}
CACHE = '/content/hf-cache'

# Everything printed below, and every error with its traceback, is also appended to a
# log on Drive so the run can be inspected without copying cell output.
LOG = ROOT / 'logs' / f'{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}-{REF[:7]}.log'
LOG.parent.mkdir(parents=True, exist_ok=True)

def log(text):
    print(text, flush=True)
    with open(LOG, 'a', encoding='utf-8') as handle:
        handle.write(text + '\n')

def call(*args):
    command = [sys.executable, '-m', 'nasn.study_v3', *map(str, args), '--cache-directory', CACHE]
    log('$ ' + ' '.join(map(str, args)))
    process = subprocess.Popen(command, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        log(line.rstrip('\n'))
    if process.wait():
        raise RuntimeError(f'Command failed with exit code {process.returncode}; see {LOG}')

import traceback
def record_error(shell, kind, value, tb, tb_offset=None):
    log('ERROR ' + ''.join(traceback.format_exception(kind, value, tb)))
    shell.showtraceback((kind, value, tb), tb_offset=tb_offset)
get_ipython().set_custom_exc((Exception,), record_error)
log(f'Run started; commit {REF}; Python {sys.version.split()[0]}')

def content_sha(path):
    return hashlib.sha256(gzip.open(path).read()).hexdigest()

import torch
GPU = torch.cuda.is_available()
log(f'GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB' if GPU
    else 'No GPU: only finished work is skipped and the analysis runs (enough once all predictions exist)')

def need_gpu(task):
    if not GPU:
        raise RuntimeError(f'{task} needs a GPU runtime (Runtime > Change runtime type > T4 GPU)')

import shutil
def drive_free_gib():
    return shutil.disk_usage('/content/drive/MyDrive').free / 2**30
# Each checkpoint needs 0.4-1.1 GiB. Below this, runs train on the Colab disk instead (see training cell).
MIN_FREE_GIB = 3
log(f'Google Drive free space: {drive_free_gib():.1f} GiB')

In [ ]:
# Test variants (must reproduce the hashes registered after the freeze).
subprocess.run([sys.executable, '-m', 'arsarcasm_audit', 'fetch', '--destination', 'data/raw'], cwd=REPO, env=ENV, check=True)
subprocess.run([sys.executable, '-m', 'nasn.arsas', 'fetch'], cwd=REPO, env=ENV, check=True)
TEST = {}
for dataset in DATASETS:
    folder = ROOT / dataset / 'test-variants'
    TEST[dataset] = folder / 'variants.csv.gz'
    if TEST[dataset].exists() and content_sha(TEST[dataset]) != TEST_VARIANTS_CONTENT_SHA256[dataset]:
        folder.rename(folder.with_name(f'test-variants-mismatch-{time.strftime("%Y%m%dT%H%M%S")}'))
    if not TEST[dataset].exists():
        call('prepare', '--dataset', dataset, '--partition', 'test', '--output', folder)
    got = content_sha(TEST[dataset])
    assert got == TEST_VARIANTS_CONTENT_SHA256[dataset], f'{dataset}: test variants {got} differ from the registered set'
    log(f'{dataset} test variants verified')

In [ ]:
# Predictions of every frozen checkpoint on the test variants.
for dataset in DATASETS:
    for model in ENCODERS:
        for seed in SEEDS:
            out = ROOT / dataset / 'test-predictions' / f'{model}-s{seed}'
            if (out / 'evidence.json').exists():
                continue
            checkpoint = ROOT / dataset / 'train' / f'{model}-s{seed}' / 'best_checkpoint'
            assert (checkpoint / 'model.safetensors').exists(), f'Missing weights: {checkpoint}'
            need_gpu(f'Test predictions {dataset}/{model}-s{seed}')
            if out.exists():
                out.rename(out.with_name(out.name + f'-incomplete-{len(list(out.parent.glob(out.name + "-incomplete-*")))}'))
            call('infer', '--variants', TEST[dataset], '--checkpoint', checkpoint, '--model', model, '--seed', seed,
                 '--defenses', *TEST_DEFENSES, '--output', out)

In [ ]:
# Test analysis and a ZIP of text-free results.
import shutil
stamp = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
public = ROOT / 'test-analysis' / f'{stamp}-test-public'
public.mkdir(parents=True)
for dataset in DATASETS:
    files = sorted(str(p) for p in (ROOT / dataset / 'test-predictions').glob('*/predictions.csv.gz') if 'incomplete' not in str(p))
    assert len(files) == len(ENCODERS) * len(SEEDS), f'{dataset}: only {len(files)} test prediction sets'
    for suffix, extra in [('all', []), ('cue-tweets', ['--cue-only'])]:
        out = ROOT / 'test-analysis' / f'{stamp}-{dataset}-{suffix}'
        call('analyze', '--variants', TEST[dataset], '--predictions', *files, *extra, '--output', out)
        for name in ['robustness_cells.csv', 'contrasts.csv', 'natural_noise_cells.csv', 'evidence.json']:
            shutil.copy(out / name, public / f'{dataset}-{suffix}-{name}')
print(shutil.make_archive(str(public), 'zip', public))